加载参数

In [11]:
from dotenv import load_dotenv

load_dotenv()

True

tool

函数集

In [12]:
# 学习资料查询tool
from pydantic import BaseModel,Field
from typing import Literal
import re

# 计算信任度的函数
def confidance(respond):
    if not respond:
        return 0.0

    scores = [score for _, score in respond]
    # 平均分
    avg_score = sum(scores)/len(scores)
    # 最高分
    max_score = max(scores)
    # 一致性
    score_range = max(scores) - min(scores)
    consistency = 1.0 - min(score_range / 0.3, 1.0)

    # 得到信任度
    confidance = avg_score * 0.4 + max_score * 0.3 + consistency * 0.3

    confidance = min(confidance,1.0)

    return confidance

# 脏数据处理函数
def clear_data(respond:str):
    lines = respond.split('\n')
    keep_lines = []
    for line in lines:
        if any(kw in line for kw in ['邮购电话', '质量投诉', '盗版侵权', 
            '咨询联系方式', '版权所有', '翻印必究',
            '联系及邮购','邮箱']):
            continue
        keep_lines.append(line)
    respond = '\n'.join(keep_lines)

    # 删除qq号、qq群号：
    pattern=r'(?:QQ|qq)[\u4e00-\u9fa5]*\s*[: ：]\s*[1-9]\d{3,12}'
    respond = re.sub(pattern, '', respond, flags=re.IGNORECASE)
    # 删除页数，章数
    pattern=r'[第]\s*\d+\s*[页|章|版]'
    respond = re.sub(pattern, '', respond, flags=re.IGNORECASE)
    # 删除邮箱
    pattern=r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    respond = re.sub(pattern, '', respond, flags=re.IGNORECASE)
    # 删除空白
    respond = re.sub(r'\s+', ' ', respond)

    respond = respond.strip()
    # respond = ''.join(respond)
    return respond

In [22]:
from langchain_core.tools import tool

@tool
def Get_Notes(user_query:str):
    '''
    - user_query:用户提问的问题
    - 每次用户提问时都检查一下回复的信任度，若信任度过低，则让用户更换询问方式
    - 每次回答都先给出信任度后在做回答，确保尽可能高的信任度
    '''
    # 检索
    try:
        if user_query is None:
            print("请输入您的问题")
        # 检索并计算信任度
        count = 1
        best_confidant = 0.0
        best_respond = None
        while count < 4:
            print(f"正在进行第{count}次检索...")
            respond = vector_db.similarity_search_with_score(user_query,k=2)
            # 检查检索结果
            if not respond or len(respond) == 0:
                return "未找到相关信息，请换个方式提问"
            confidant = confidance(respond)# 计算

            if confidant > best_confidant:# 找到最佳回答
                best_confidant = confidant
                best_respond = respond
            count += 1
        # 处理脏数据
        raw_content = best_respond[0][0].page_content
        clear_respond = clear_data(raw_content)
        # clear_respond = 1
    
        if best_confidant >= 0.7:
            result= f"置信度是{best_confidant * 100}%,可以相信,内容是{clear_respond}"
        elif best_confidant >= 0.4:
            result= f"置信度是{best_confidant * 100}%,置信度较低,请谨慎相信此答案,内容是{clear_respond}"
        else:
            result= f"置信度是{best_confidant * 100}%,无法找到答案，请换个询问方式"

        return result
    except Exception as e:
        return f"检索失败:{e}"

##### memory系统

In [14]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from datetime import datetime,timedelta

# 这里指定记忆缓存的位置
connection = sqlite3.connect("./SQlite/checkpoint.db",check_same_thread=False)
checkpointer = SqliteSaver(connection)# 初始化checkpointer
checkpointer.setup()

config = {"configurable":{"thread_id": "thread_1"}}

# 自动清理函数
def clean_thread(thread_id_to_delete):
    """删除特定会话的所有历史"""
    try:
        cursor = connection.cursor()
        cursor.execute(
            "DELETE FROM checkpoints WHERE thread_id = ?",
            (thread_id_to_delete,)
        )
        cursor.execute(
            "DELETE FROM writes WHERE thread_id = ?",  # 同时删除关联的 writes 表
            (thread_id_to_delete,)
        )
        connection.commit()
        print(f"已删除 thread_id: {thread_id_to_delete}")
    except Exception as e:
        print(f"删除失败: {e}")

##### 上下文窗口

In [15]:
from langchain.agents.middleware import SummarizationMiddleware

middleware = SummarizationMiddleware(
    model="deepseek-v4-pro",# 让模型自己给超出的文本进行总结
    trigger=("messages",6),
    keep=("messages",3)
)

##### prompt

In [16]:
system_prompt = '''
# 身份
你是一个日常学习助手，帮助用户查找并总结知识库中的知识点。

# 工具使用说明
- 当用户询问计算机相关知识时，使用 Get_Notes 工具检索知识库
- 工具会返回置信度信息，请根据置信度处理回答

# 置信度处理规则

## 高置信度（>=70%）
直接采用检索结果回答用户，语气确定。

## 中等置信度（40-69%）
采用检索结果，但要提醒用户信息可能不够完整：
"根据检索结果...建议您核实..."

## 低置信度（<40%）
不要直接回答，而是引导用户：
"抱歉，我对这个问题不太确定。建议您..."
如果用户问题在知识库之外，可以用自身知识回答。

# 多义词处理
当用户问题存在多个含义时（如"苹果"可指水果或公司）：
1. 列出所有可能的含义
2. 询问用户具体想了解哪一种
3. 根据用户选择提供详细信息

# 一般情况
- 如果用户问题在知识库之外，可以用自身知识回答
- 如果完全不知道，诚实告知

# 输出格式
- 必须使用 Markdown 格式
- 包含置信度说明（如有）
- 保持友好、专业的语气
'''

##### 加载向量库

In [17]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 初始化embedding模型
emd = HuggingFaceEmbeddings(
    model_name = "D:/AI_tool/embedding_models/BAAI/models/BAAI--bge-base-zh-v1.5/snapshots/master",
    model_kwargs={"device": "cpu"}
)

# 加载数据库
vector_db = Chroma(
    persist_directory = "./local_doc_db",
    embedding_function=emd
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

调用大模型

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-v4-pro",
    system_prompt=system_prompt,
    checkpointer=checkpointer,
    middleware=[middleware],
    tools=[Get_Notes]
)

输出

##### 交互式输出

In [23]:
from langchain.messages import HumanMessage

if __name__ == "__main__":
    print("学习聊天助手已启动，输入exit退出程序")
    enquire_count = 0
    while True:
        enquire_count += 1
        print("=" * 40 + f"第{enquire_count}段对话" + "=" * 40)
        user_query = input("输入")
        print(f"user:{user_query}\n")

        if user_query.strip().lower() == "exit":
            print("程序退出")
            clean_thread("thread_1")# 删除本次的记忆
            break


        # 回复
        agent_response = agent.invoke({
            "messages":[HumanMessage(user_query)]},
            config
            )
        print(f"AI:{agent_response['messages'][-1].content}\n") 

学习聊天助手已启动，输入exit退出程序
========================================第1段对话========================================
user:你好

AI:你好！😊 我是你的日常学习助手，可以帮你查找和总结计算机相关的知识点。

有什么我可以帮你的吗？比如：
- 想了解某个计算机概念或技术？
- 需要复习某个知识点？
- 有其他学习相关的问题？

随时告诉我，我会尽力帮你解答！

========================================第2段对话========================================
user:什么是计算机网络

AI:## 什么是计算机网络

**置信度：60.72%（中等置信度）** ⚠️

根据检索结果，以下内容供您参考，建议您结合实际教材进一步核实：

---

### 计算机网络的定义

计算机网络的精确定义并未完全统一，一个较好的定义是（[PETE11]）：

> **计算机网络主要是由一些通用的、可编程的硬件互连而成的，而这些硬件并非专门用来实现某一特定目的（例如，传送数据或视频信号）。**

这些可编程的硬件能够用来传送多种不同类型的数据，并能支持广泛的和日益增长的应用。

### 根据这个定义，有两点值得注意：

1. **连接的硬件不限于一般计算机** —— 也包括了智能手机等设备；
2. **网络并非只用来传送数据** —— 而是能够支持很多种应用（包括今后可能出现的各种应用）。

### 补充说明

- 这里的 **"可编程的硬件"** 表明这种硬件一定包含有**中央处理机 CPU**。
- 起初，计算机网络的确是用来传送数据的，但随着网络技术的发展，其应用范围已经大大扩展。

---

💡 **小提示**：由于本次检索置信度处于中等水平，以上信息可能不够完整。如果您需要更全面、权威的定义，建议查阅《计算机网络》相关教材（如谢希仁版或 Andrew S. Tanenbaum 版）第一章的内容。

请问您是想了解计算机网络的定义、分类，还是其他具体方面呢？我可以进一步帮您查找～

========================================第3段对话===============